# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
import dask.dataframe as dd

# Write your code below.
price_data_dr = os.getenv("PRICE_DATA")
if price_data_dr is None:
    raise RuntimeError("PRICE_DATA environment variable is not set")
parquet_files = glob(
    os.path.join(price_data_dr, "**", "*.parquet"),
    recursive=True
)

print(len(parquet_files))



2756


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
# Write your code below.
import dask.dataframe as dd
import pandas as pd

# Reading Parquet data
dd_px = dd.read_parquet(parquet_files).set_index('ticker')

meta = dd_px._meta.assign(
    Close_lag_1=pd.Series(dtype='float64'),
    Adj_Close_lag_1=pd.Series(dtype='float64')
)
dd_feat = (
    dd_px
    .groupby('ticker', group_keys=False)
    .apply(
        lambda x: x.sort_values('Date', ascending=True).assign(
            Close_lag_1=lambda df:df['Close'].shift(1),
            Adj_Close_lag_1=lambda df:df['Adj Close'].shift(1)
        ),
        meta=meta
    )
    .assign(
        returns=lambda x: x['Close'] / x['Close_lag_1'] - 1,
        hi_lo_range=lambda x: x['High'] - x['Low']
    )
)



+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [5]:
# Write your code below.
# Computing 10-observation moving avg returns
df_feat = dd_feat.compute()
df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].transform(
    lambda x: x.rolling(10).mean()
    )
df_feat.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
ticker,,,,,,,,,,,,,,
A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN,NaN,7.153078,NaN
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,2.280043,NaN
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,2.816525,NaN
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,2.592991,NaN
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,1.385908,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, It is not necessary. Dask supports rolling mean function and it could have been computed before calling compute.
However,Dask rolling window operates within each partition, not across partition boundaries.
Since the data is partitioned by ticker/year, a rolling window spanning year boundary would produce NaN values at the start of each new year partition.
Converting to pandas first and then grouping by ticker ensures the rolling window sees full chronological sequence for each ticker without partition boundary issues.

For small datasets like this one (30 tickers), pandas  is fine. 
For large datasets like thousands of tickers that will not fit in memory, we would need to use in Dask and handle this partition boundary problem explicitly.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.